# Практическая работа 4. Advanced RAG

Выполнены задания:

1. Эксперимент с Cross-Encoder (reranker).
2. Настройка гиперпараметров гибридного поиска.

Схема RAG такая же, как на занятии: BM25 и векторный поиск объединяются через `EnsembleRetriever`, затем документы при необходимости переранжируются, и языковая модель отвечает по найденному контексту.

Используемые модели работают локально:
- языковая модель SmolLM2-135M-Instruct через `llama-cpp-python`;
- кодировщики текста: WordLlama (небольшая модель эмбеддингов) и TF-IDF из `scikit-learn`.

Модели Cross-Encoder с Hugging Face (`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, `cross-encoder/ms-marco-MiniLM-L-6-v2`) скачать не удалось, поэтому в роли Cross-Encoder используется языковая модель: она, как и Cross-Encoder, получает вопрос и документ вместе и выдает оценку их связи. Сравниваются два таких reranker'а с разным способом оценки.

Модель понимает в основном английский язык, поэтому база знаний и вопросы на английском.

In [1]:
# pip install llama-cpp-python llm-smollm2 wordllama scikit-learn rank_bm25 langchain-core langchain-community langchain-classic pandas numpy

In [2]:
import os, time, json, ctypes, importlib.util, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import llama_cpp
from llama_cpp import Llama
import wordllama
from wordllama import WordLlama
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

@llama_cpp.llama_log_callback
def quiet_log(level, text, user_data):
    pass
llama_cpp.llama_log_set(quiet_log, ctypes.c_void_p())

MODEL_PATH = os.path.join(importlib.util.find_spec("llm_smollm2").submodule_search_locations[0],
                          "SmolLM2-135M-Instruct.Q4_1.gguf")
# logits_all нужен, чтобы получать вероятности токенов для reranker'а
llm = Llama(MODEL_PATH, n_ctx=2048, logits_all=True, verbose=False)

## Данные
Небольшая база знаний о прокате велосипедов (20 документов) и 8 вопросов. Для каждого вопроса известен правильный документ.

In [3]:
DOCS = [
    "Northwind Bike Rental is open every day from 8 am to 9 pm. In winter, from December to February, it closes at 6 pm.",
    "A city bike costs 10 dollars per hour or 30 dollars per day.",
    "Electric bikes cost 15 dollars per hour or 45 dollars per day. The battery lasts for about 60 kilometers.",
    "To rent a bike you need a passport or a driver's license. Children under 16 can rent only with a parent.",
    "A deposit of 100 dollars is required for every bike. The deposit is returned when the bike comes back undamaged.",
    "Helmets are free for all customers. Child seats cost 5 dollars per day.",
    "If a bike is returned late, the customer pays a fee of 8 dollars for every extra hour.",
    "Customers can cancel a booking for free up to 24 hours before the start. Later cancellations cost 50 percent of the price.",
    "Every bike is checked by a mechanic after each rental. Brakes and tires are replaced every three months.",
    "If a bike breaks during the trip, call the support line and a mechanic will arrive within 40 minutes.",
    "Students get a 20 percent discount on weekdays. A student card must be shown.",
    "Groups of five or more people get a 15 percent discount on any day.",
    "Payment is accepted by credit card or cash. Mobile payments are not accepted yet.",
    "The rental has three points: near the central station, in the old town and at the river park.",
    "A bike can be returned to any of the three rental points without extra charge.",
    "Guided tours start every Saturday at 10 am from the old town point. A tour lasts three hours.",
    "Dogs are allowed in the bike basket if they weigh less than 8 kilograms.",
    "Lost or stolen bikes must be reported to the police. The customer pays the full price of the bike if it is not found.",
    "Locks are included with every bike. The lock code is printed on the rental receipt.",
    "Gift cards can be bought in all rental points and are valid for one year.",
]
documents = [Document(page_content=t, metadata={"id": i}) for i, t in enumerate(DOCS)]

QUESTIONS = [
    {"q": "How much does an electric bike cost for a whole day?", "doc": 2},
    {"q": "What documents do I need to take a bike?", "doc": 3},
    {"q": "How much is the penalty for bringing the bike back after the time ends?", "doc": 6},
    {"q": "Can I cancel my reservation without paying?", "doc": 7},
    {"q": "What should I do if my bicycle stops working on the road?", "doc": 9},
    {"q": "Is there a lower price for university students?", "doc": 10},
    {"q": "Can I pay with my phone?", "doc": 12},
    {"q": "When does the guided tour begin?", "doc": 15},
]

## Кодировщики текста
Для векторного поиска нужны классы в формате LangChain `Embeddings`.

In [4]:
class WordLlamaEmbeddings(Embeddings):
    def __init__(self):
        # файлы модели лежат внутри пакета wordllama
        self.model = WordLlama.load(cache_dir=Path(wordllama.__file__).parent, disable_download=True)
    def embed_documents(self, texts):
        return self.model.embed(texts, norm=True).tolist()
    def embed_query(self, text):
        return self.embed_documents([text])[0]

class TfidfEmbeddings(Embeddings):
    def __init__(self, corpus):
        self.vectorizer = TfidfVectorizer(stop_words="english").fit(corpus)
    def embed_documents(self, texts):
        # малая добавка, чтобы у вопроса без известных слов вектор не был нулевым
        return (self.vectorizer.transform(texts).toarray() + 1e-9).tolist()
    def embed_query(self, text):
        return self.embed_documents([text])[0]

ENCODERS = {"WordLlama": WordLlamaEmbeddings(), "TF-IDF": TfidfEmbeddings(DOCS)}
stores = {name: InMemoryVectorStore.from_documents(documents, emb) for name, emb in ENCODERS.items()}

def make_retriever(encoder="WordLlama", weights=(0.5, 0.5), k=4):
    bm25 = BM25Retriever.from_documents(documents, k=k)
    vector = stores[encoder].as_retriever(search_kwargs={"k": k})
    return EnsembleRetriever(retrievers=[bm25, vector], weights=list(weights))

## Генерация ответа и метрики
- `hit@3`: правильный документ попал в первые три;
- `MRR`: среднее значение 1/позиция правильного документа (1, если он первый);
- ответы модели проверены вручную по тексту документов.

In [5]:
def answer(question, docs):
    context = "\n".join(d.page_content for d in docs)
    r = llm.create_chat_completion(messages=[
        {"role": "system", "content": "Answer the question using only the context. Answer in one sentence."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
        max_tokens=60, temperature=0, seed=0)
    return r["choices"][0]["message"]["content"].strip()

def rank_of(docs, doc_id):
    ids = [d.metadata["id"] for d in docs]
    return ids.index(doc_id) + 1 if doc_id in ids else None

def retrieval_metrics(ranks):
    return {"hit@3": np.mean([r is not None and r <= 3 for r in ranks]),
            "MRR": np.mean([1 / r if r else 0 for r in ranks])}

## Задание 1. Cross-Encoder (reranker)

Два reranker'а на основе языковой модели:
- **A (да/нет)**: модели дают документ и вопрос и спрашивают, отвечает ли документ на вопрос. Оценка равна вероятности ответа «yes».
- **B (правдоподобие вопроса)**: оценивается, насколько вероятен текст вопроса после текста документа. Оценка равна средней логарифмической вероятности токенов вопроса.

Пайплайн: гибридный поиск возвращает до 8 кандидатов, reranker сортирует их, в контекст идут первые 3.

In [6]:
def rerank_yes_no(question, docs):
    scores = []
    for d in docs:
        r = llm.create_chat_completion(messages=[
            {"role": "user", "content": f"Document: {d.page_content}\nQuestion: {question}\n"
                                        "Does the document answer the question? Reply yes or no."}],
            max_tokens=1, temperature=0, logprobs=True, top_logprobs=20)
        top = r["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
        p = {"yes": 0.0, "no": 0.0}
        for t in top:
            word = t["token"].strip().lower()
            if word in p:
                p[word] += np.exp(t["logprob"])
        scores.append(p["yes"] / (p["yes"] + p["no"] + 1e-9))
    return scores

def rerank_likelihood(question, docs):
    scores = []
    q_tokens = llm.tokenize((" " + question).encode(), add_bos=False)
    for d in docs:
        prefix = llm.tokenize(f"Document: {d.page_content}\nQuestion:".encode())
        llm.reset()
        llm.eval(prefix + q_tokens)
        logits = np.array(llm.scores[len(prefix) - 1: len(prefix) + len(q_tokens) - 1])
        m = logits.max(1, keepdims=True)
        logprobs = logits - m - np.log(np.exp(logits - m).sum(1, keepdims=True))
        scores.append(float(np.mean(logprobs[np.arange(len(q_tokens)), q_tokens])))
    return scores

RERANKERS = {"без reranker": None, "A (да/нет)": rerank_yes_no, "B (правдоподобие)": rerank_likelihood}

base = make_retriever(k=4)
results = []
for item in QUESTIONS:
    candidates = base.invoke(item["q"])
    for name, func in RERANKERS.items():
        t = time.perf_counter()
        if func:
            scores = func(item["q"], candidates)
            ranked = [d for _, d in sorted(zip(scores, candidates), key=lambda x: -x[0])]
        else:
            ranked = candidates
        t_rerank = time.perf_counter() - t
        top3 = ranked[:3]
        t = time.perf_counter()
        ans = answer(item["q"], top3)
        results.append({"вопрос": item["q"], "reranker": name, "порядок": [d.metadata["id"] for d in ranked],
                         "позиция нужного": rank_of(ranked, item["doc"]), "ответ": ans,
                         "время rerank, с": t_rerank, "время ответа, с": time.perf_counter() - t})
res1 = pd.DataFrame(results)

### 1.1. Порядок документов после reranking
В скобках номер правильного документа. Показан порядок кандидатов (номера документов).

In [7]:
order = res1.pivot(index="вопрос", columns="reranker", values="порядок")
order.insert(0, "нужный", [next(x["doc"] for x in QUESTIONS if x["q"] == q) for q in order.index])
order

reranker,нужный,A (да/нет),B (правдоподобие),без reranker
вопрос,,,,
Can I cancel my reservation without paying?,7,"[7, 12, 18, 19, 14, 6]","[7, 12, 19, 6, 14, 18]","[7, 14, 12, 19, 18, 6]"
Can I pay with my phone?,12,"[12, 7, 19, 18, 17, 6, 3]","[12, 6, 7, 17, 19, 18, 3]","[17, 18, 12, 3, 6, 7, 19]"
How much does an electric bike cost for a whole day?,2,"[5, 7, 2, 6, 1, 8]","[2, 1, 6, 5, 8, 7]","[2, 6, 5, 7, 1, 8]"
How much is the penalty for bringing the bike back after the time ends?,6,"[14, 4, 6, 5, 17, 8]","[6, 4, 14, 17, 5, 8]","[4, 6, 8, 14, 5, 17]"
Is there a lower price for university students?,10,"[7, 11, 10, 17, 5, 6, 3]","[10, 7, 6, 5, 3, 17, 11]","[7, 17, 10, 6, 11, 3, 5]"
What documents do I need to take a bike?,3,"[0, 14, 17, 7, 4, 8, 3]","[3, 8, 14, 17, 4, 0, 7]","[3, 4, 7, 0, 17, 14, 8]"
What should I do if my bicycle stops working on the road?,9,"[4, 18, 9, 17, 11, 16, 10, 8]","[8, 9, 18, 4, 17, 16, 10, 11]","[16, 8, 10, 9, 11, 17, 18, 4]"
When does the guided tour begin?,15,"[7, 19, 9, 10, 15, 18, 17]","[15, 19, 9, 18, 17, 7, 10]","[15, 19, 9, 17, 7, 18, 10]"


### 1.2. Ответы модели

In [8]:
pd.set_option("display.max_colwidth", 120)
res1.pivot(index="вопрос", columns="reranker", values="ответ")

reranker,A (да/нет),B (правдоподобие),без reranker
вопрос,,,
Can I cancel my reservation without paying?,The answer is no.,"Yes, you can cancel a reservation without paying.","Yes, you can cancel your reservation without paying."
Can I pay with my phone?,"Yes, you can pay with your phone.","Yes, you can pay with your phone.","Lost or stolen bikes can be reported to the police, and the bike must be found. The customer pays the full price of ..."
How much does an electric bike cost for a whole day?,The answer is: 15.55.,A single electric bike costs 15 dollars per hour or 45 dollars per day.,A single electric bike can cost around 15 dollars per hour or 45 dollars per day.
How much is the penalty for bringing the bike back after the time ends?,The penalty for bringing the bike back after the time ends is 8 dollars.,The penalty for bringing the bike back after the time ends is 100 dollars.,The penalty for bringing the bike back after the time ends is 10 dollars.
Is there a lower price for university students?,"The answer to this question is no, there is no lower price for university students.","The answer is yes, there is a lower price for university students.","The answer is no, there is no lower price for university students."
What documents do I need to take a bike?,"To ensure you have all the necessary documents to ride a Northwind Bike, you'll need to bring a valid bike rental ca...","You need a valid passport or driver's license to rent a bike, and a mechanic to replace your tires.","To rent a bike, you need a passport or a driver's license, and a deposit of 100 dollars."
What should I do if my bicycle stops working on the road?,"If your bike stops working on the road, it's best to contact the support line and a mechanic to have it repaired.","If your bike stops working on the road, it's best to contact the repair service immediately.","If your bicycle stops working on the road, you should take it to a local bike shop for a repair."
When does the guided tour begin?,The guided tour begins 24 hours before the start date.,The guided tour begins at 10 am on Saturday at the old town point.,The guided tour begins at 10 am on Saturday at the old town point.


Ручная проверка ответов (сколько из 8 ответов верные по смыслу):

| Вариант | Верных ответов | Типичные ошибки |
|---|---|---|
| без reranker | 4 | неверный штраф (10 вместо 8), ответ про оплату телефоном не по теме |
| A (да/нет) | 2 | неверная цена, неверное время экскурсии, отказ в скидке студентам |
| B (правдоподобие) | 6 | неверный штраф (100 вместо 8), «можно платить телефоном» |

### 1.3. Сводная таблица

In [9]:
summary1 = res1.groupby("reranker", sort=False).agg(
    hit3=("позиция нужного", lambda r: np.mean([x is not None and x <= 3 for x in r])),
    MRR=("позиция нужного", lambda r: np.mean([1 / x if x else 0 for x in r])),
    rerank_с=("время rerank, с", "mean"),
    ответ_с=("время ответа, с", "mean")).round(3)
summary1

,hit3,MRR,rerank_с,ответ_с
reranker,,,,
без reranker,0.875,0.677,0.000,0.519
A (да/нет),0.750,0.460,1.713,0.484
B (правдоподобие),1.000,0.938,0.809,0.492


**Вывод по заданию 1.** Выбор reranker'а сильно влияет на результат. Reranker B поднял правильный документ на первое место в 7 вопросах из 8: `MRR` вырос с 0,68 до 0,94, а `hit@3` до 1,0. Вместе с контекстом улучшились и ответы (6 верных из 8 против 4 без reranker'а). Reranker A, наоборот, ухудшил порядок (`MRR` 0,46) и ответы (2 из 8): маленькая модель плохо отвечает «да» или «нет» на вопрос о релевантности. Reranking требует дополнительного времени (около 0,8 с для B и 1,7 с для A на вопрос при 6–8 кандидатах), что больше времени генерации ответа. Вывод: reranker полезен, только если сама модель-оценщик хорошая, поэтому его нужно проверять на своих данных. Даже при правильном контексте маленькая модель иногда ошибается в числах, то есть качество ответа ограничено и генератором.

## Задание 2. Гиперпараметры гибридного поиска

Меняются:
- веса `EnsembleRetriever` (первый вес для BM25, второй для векторного поиска): `[0.5, 0.5]`, `[0.7, 0.3]`, `[0.3, 0.7]`;
- кодировщик для векторного поиска: WordLlama или TF-IDF;
- число документов `k` у каждого ретривера: 2 или 4.

Для каждой комбинации прогоняются все 8 вопросов. Сначала оценивается только поиск.

In [10]:
from collections import Counter

rows, chosen = [], {}
for encoder in ENCODERS:
    for weights in [(0.5, 0.5), (0.7, 0.3), (0.3, 0.7)]:
        for k in [2, 4]:
            retriever = make_retriever(encoder, weights, k)
            ranks, times, top_ids = [], [], Counter()
            for item in QUESTIONS:
                t = time.perf_counter()
                docs = retriever.invoke(item["q"])
                times.append(time.perf_counter() - t)
                ranks.append(rank_of(docs, item["doc"]))
                top_ids.update(d.metadata["id"] for d in docs[:3])
            key = (encoder, str(list(weights)), k)
            chosen[key] = top_ids
            rows.append({"кодировщик": encoder, "веса": str(list(weights)), "k": k,
                         **retrieval_metrics(ranks), "поиск, мс": np.mean(times) * 1000})
grid = pd.DataFrame(rows).round(3)
grid.sort_values(["hit@3", "MRR"], ascending=False)

,кодировщик,веса,k,hit@3,MRR,"поиск, мс"
4,WordLlama,"[0.3, 0.7]",2,1.000,0.938,0.621
5,WordLlama,"[0.3, 0.7]",4,1.000,0.812,0.624
1,WordLlama,"[0.5, 0.5]",4,0.875,0.677,0.664
0,WordLlama,"[0.5, 0.5]",2,0.875,0.656,0.855
2,WordLlama,"[0.7, 0.3]",2,0.875,0.573,0.728
11,TF-IDF,"[0.3, 0.7]",4,0.750,0.604,1.090
7,TF-IDF,"[0.5, 0.5]",4,0.750,0.542,1.087
3,WordLlama,"[0.7, 0.3]",4,0.625,0.571,0.642
9,TF-IDF,"[0.7, 0.3]",4,0.625,0.525,1.110
6,TF-IDF,"[0.5, 0.5]",2,0.625,0.500,1.107


### 2.1. Какие документы выбираются чаще
Для каждого кодировщика показано, сколько раз документ попадал в первые три (сумма по всем весам и `k`). Правильных документов 8, остальные 12 лишние.

In [11]:
gold = {x["doc"] for x in QUESTIONS}
freq = []
for encoder in ENCODERS:
    total = Counter()
    for key, cnt in chosen.items():
        if key[0] == encoder:
            total.update(cnt)
    useful = sum(v for d, v in total.items() if d in gold)
    freq.append({"кодировщик": encoder, "самые частые документы": total.most_common(5),
                 "доля правильных документов в топ-3": round(useful / sum(total.values()), 2)})
pd.DataFrame(freq)

,кодировщик,самые частые документы,доля правильных документов в топ-3
0,WordLlama,"[(7, 18), (6, 14), (4, 11), (17, 11), (12, 10)]",0.55
1,TF-IDF,"[(7, 34), (2, 16), (14, 12), (3, 10), (4, 10)]",0.59


### 2.2. Ответы модели для разных настроек
Генерация запускается для трех настроек: исходной (`[0.5, 0.5]`, WordLlama, k=4), лучшей и худшей по `MRR`.

In [12]:
ordered = grid.sort_values(["MRR", "hit@3"], ascending=False)
configs = {"исходная": ("WordLlama", "[0.5, 0.5]", 4),
           "лучшая": (ordered.iloc[0]["кодировщик"], ordered.iloc[0]["веса"], int(ordered.iloc[0]["k"])),
           "худшая": (ordered.iloc[-1]["кодировщик"], ordered.iloc[-1]["веса"], int(ordered.iloc[-1]["k"]))}
print(configs)

rows = []
for label, (encoder, weights, k) in configs.items():
    retriever = make_retriever(encoder, json.loads(weights), int(k))
    for item in QUESTIONS:
        t = time.perf_counter(); docs = retriever.invoke(item["q"])[:3]; t_search = time.perf_counter() - t
        t = time.perf_counter(); ans = answer(item["q"], docs); t_gen = time.perf_counter() - t
        rows.append({"настройка": label, "вопрос": item["q"], "ответ": ans,
                     "поиск, мс": t_search * 1000, "генерация, с": t_gen})
res2 = pd.DataFrame(rows)
res2.pivot(index="вопрос", columns="настройка", values="ответ")

{'исходная': ('WordLlama', '[0.5, 0.5]', 4), 'лучшая': ('WordLlama', '[0.3, 0.7]', 2), 'худшая': ('TF-IDF', '[0.7, 0.3]', 2)}


настройка,исходная,лучшая,худшая
вопрос,,,
Can I cancel my reservation without paying?,"Yes, you can cancel your reservation without paying.","Yes, you can cancel your reservation without paying.","A customer can cancel a reservation for free up to 24 hours before the start, but can also cancel a reservation for ..."
Can I pay with my phone?,"Lost or stolen bikes can be reported to the police, and the bike must be found. The customer pays the full price of ...","Yes, you can pay with your phone.",The answer is yes. The answer is yes. The answer is no. The answer is no. The answer is no. The answer is no. The an...
How much does an electric bike cost for a whole day?,A single electric bike can cost around 15 dollars per hour or 45 dollars per day.,A city bike costs 15 dollars per hour or 45 dollars per day.,The answer is: 15.55.
How much is the penalty for bringing the bike back after the time ends?,The penalty for bringing the bike back after the time ends is 10 dollars.,The penalty for bringing the bike back after the time ends is 100 dollars.,The penalty for bringing the bike back after the time ends is 10% of the rental fee.
Is there a lower price for university students?,"The answer is no, there is no lower price for university students.","The answer to this question is no, there is no lower price for university students.","The answer is no, there is no lower price for university students."
What documents do I need to take a bike?,"To rent a bike, you need a passport or a driver's license, and a deposit of 100 dollars.","To rent a bike, you need a passport or a driver's license, and a deposit of 100 dollars.","To rent a bike, you need a valid passport or driver's license, a valid passport or driver's license, and a deposit o..."
What should I do if my bicycle stops working on the road?,"If your bicycle stops working on the road, you should take it to a local bike shop for a repair.","If your bike stops working on the road, it's best to call the local emergency services to report the issue.","If your bicycle stops working on the road, you should consider the situation carefully and take necessary steps to e..."
When does the guided tour begin?,The guided tour begins at 10 am on Saturday at the old town point.,The guided tour begins at 10 am on Saturday from the old town point.,The guided tour begins at 10 am on Saturday from the old town point.


Ручная проверка ответов: исходная настройка 4 верных из 8, лучшая 4 из 8, худшая 3 из 8.

In [13]:
res2.groupby("настройка", sort=False).agg(поиск_мс=("поиск, мс", "mean"),
                                          генерация_с=("генерация, с", "mean")).round(3)

,поиск_мс,генерация_с
настройка,,
исходная,1.399,0.510
лучшая,1.329,0.427
худшая,1.775,0.559


**Вывод по заданию 2.** Лучшие результаты поиска дала комбинация WordLlama, веса `[0.3, 0.7]` (больший вес у векторного поиска) и `k=2`: правильный документ попал в топ-3 во всех вопросах, `MRR` 0,94. Увеличение веса BM25 ухудшает поиск, так как в вопросах часто используются другие слова, чем в документах (например, «penalty» и «fee», «phone» и «mobile payments»). TF-IDF работает хуже WordLlama по той же причине: он учитывает только совпадение слов. Меньшее `k` дало более точный порядок, так как в ансамбль попадает меньше лишних документов. Чаще всего в топ-3 попадает документ об отмене брони (№7): в нем есть частые в вопросах слова «cost», «price» и «free». Время поиска на такой базе около 1–2 мс для всех настроек и почти не зависит от параметров, а генерация занимает около 0,5 с. Лучший поиск не дал заметного улучшения ответов (4 верных из 8, как у исходной), так как маленькая модель ошибается даже при правильном контексте. Оптимальный баланс: больший вес у векторного поиска с хорошим кодировщиком и небольшое `k`.